# ByteEmbed — FINAL retrieval study (BGE-M3 teacher)

The finalized retrieval/QA experiment. Identical to the SONAR study in every respect **except the
teacher**: students distill **BGE-M3** (`BAAI/bge-m3`, retrieval-trained, 1024-d, 100+ languages)
instead of SONAR (a bitext encoder). One variable changes vs the SONAR run, so the two runs are
directly comparable — and the deep-retrieval ceiling (which was the teacher, not the students) lifts.

**Design (locked):**
- **Students:** byt5 vs mt5 × {small, base, large} — 6 models
- **Recipe:** InfoNCE (τ=0.05, queue 8192) + alignment + relational, AdamW lr 2e-4, **batch 64,
  50k steps for every model (iso-step)**, `attn` pooling for all (fair)
- **Data:** same 9 languages, same balanced ~42k sentences/lang (~378k total) — unchanged
- **Teacher targets:** BGE-M3, cached once (`teachertargets_bge-m3_9langs_42000.npy`)
- **Headline benchmarks (nDCG@10):** Belebele (all 9) · MIRACL (en/zh/ar/te) · IndicQA (mr/ta/te) ·
  Amharic-PR (am) · 2AIRTC (am); Mr.TyDi + AfriCLIR still computed, reported as secondary
- **Baselines:** mE5-base, LaBSE (same battery)

See `RETRIEVAL_EXPERIMENT.md` (repo root) for the full protocol + rationale. Everything is resumable;
run top-to-bottom; smoke first.

### 1. GPU check — confirm you're on an A100 (Runtime → Change runtime type → A100)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')

### 2. Clone repo + install deps
Imports work from the repo root even if the editable install is skipped.

In [ ]:
import os
os.chdir('/content')
REPO = 'https://github.com/Aarushvinod/embedding-research.git'
if not os.path.isdir('/content/embedding-research'):
    !git clone -q $REPO
os.chdir('/content/embedding-research')
!git pull -q
!pip install -q -r requirements-cloud.txt
!pip install -q -e . || echo '(editable install skipped — running from repo root is fine)'
print('setup done | cwd', os.getcwd())

### 3. Teacher check + persist to Drive
BGE-M3 loads via sentence-transformers (already a dep) — no extra install, no fairseq2. Point
`PERSIST` at the **same** `byteembed_lowres` folder as the SONAR runs: the balanced-data cache is
reused; the BGE-M3 teacher targets get their own cache file (`teachertargets_bge-m3_*`), so nothing
collides. Skip the Drive block to run on ephemeral disk.

In [ ]:
from huggingface_hub import hf_hub_download
_ = hf_hub_download('BAAI/bge-m3', 'config.json')   # reachable? (weights download on first teacher load)
print('BGE-M3 reachable — teacher will be BAAI/bge-m3 (retrieval-trained, 1024-d)')

from google.colab import drive
drive.mount('/content/drive')
import os, shutil
PERSIST = '/content/drive/MyDrive/byteembed_lowres'   # SAME folder as the SONAR runs -> shared data cache
for d in ('results', 'checkpoints'):
    os.makedirs(f'{PERSIST}/{d}', exist_ok=True)
    if not os.path.islink(d):
        if os.path.isdir(d): shutil.rmtree(d)
        os.symlink(f'{PERSIST}/{d}', d)
print('persisting results/ and checkpoints/ to', PERSIST)

### 4. Smoke test (~5 min) — validate the pipeline with the NEW teacher
3 langs (am/rw/en), 2 tiny students, tiny eval. Confirms: BGE-M3 loads → targets precompute + cache →
train → full eval battery (incl. QA-retrieval) → save. **Check the log says 'BGE-M3 ... loaded' — not
SONAR, not LaBSE.**

In [ ]:
from byte_embed.run_lowresource import run
_ = run(smoke=True, teacher_name='bge-m3', pooling='attn', out='results/retrieval_bgem3_smoke.json')

### 5. Full retrieval study — 6 students, BGE-M3 targets
**50k steps × batch 64 for every model (iso-step), `attn` pooling for all.** The parallel runner
precomputes the BGE-M3 targets once, then trains several students at once. Resumable — finished
models skip; checkpoints carry a `_bge-m3` suffix (`byte-small_attn_bge-m3.pt`) so they never collide
with — or wrongly resume from — the SONAR-run checkpoints.

In [ ]:
from byte_embed.run_parallel import parallel
parallel(
    out='results/retrieval_bgem3.json',
    teacher_name='bge-m3',            # THE one change vs the SONAR study
    pooling='attn',                   # same pooling for byte AND subword (fair)
    steps=50000,                      # iso-step: 50k for every size
    max_concurrent=3,                 # 80/96 GB card: 3-5
)

# --- sequential fallback (one model at a time, live logs in-cell):
# from byte_embed.run_lowresource import run
# _ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 6. Baselines + summary
The parallel runner trains students only. This cell adds the mE5-base + LaBSE baselines to the same
results file (students all skip — already present) and prints the full table incl. the RAG-retrieval
block (IndicQA · Mr.TyDi · Amharic-PR · 2AIRTC · AfriCLIR).

In [ ]:
from byte_embed.run_lowresource import run
_ = run(out='results/retrieval_bgem3.json', teacher_name='bge-m3', pooling='attn', steps=50000)

### 7. Download results
Already on Drive if you ran the persist cell; otherwise grab the JSON here.

In [ ]:
from google.colab import files
files.download('results/retrieval_bgem3.json')